In [0]:
from TornAPI.Torn import Faction
from pyspark.sql.functions import lit

import datetime as dt

faction_api = Faction(dbutils.secrets.get("Personal", "TornAPI"))

In [0]:
if spark.catalog.tableExists("torn.faction.basics"):
    df = spark.read.table("torn.faction.basics")
    last_date = df.select("date").agg({"date": "max"}).collect()[0].asDict()["max(date)"]
else:
    last_date = dt.date.today() + dt.timedelta(days=-1)

In [0]:
if dt.date.today() > last_date:

    basic_data = faction_api.get_basic()

    sp_basic_data = spark.createDataFrame([basic_data["basic"]])
    sp_basic_data = sp_basic_data.withColumn("date", lit(dt.date.today()))

    sp_basic_data.write.format("delta").mode("append").saveAsTable("torn.faction.basics")